In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

X = pd.read_csv("train.csv")
y = X.pop("label").to_numpy(dtype=np.int64)

X = X.to_numpy(dtype=np.float32).reshape(-1, 1, 28, 28) / 255.0

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
from modules.conv import *
from modules.flatten import *
from modules.linear import *
from modules.maxpool import *
from modules.relu import *

class CNN:
    def __init__(self):
        self.sequence = [
            Conv2D(1, 8, 3, 1),
            ReLU(),
            MaxPool2D(2),

            Conv2D(8, 16, 3, 1),
            ReLU(),
            MaxPool2D(2),

            Flatten(),
            Linear(16*7*7, 128),
            ReLU(),
            Linear(128, 10)
        ]

    def forward(self, x):
        for layer in self.sequence:
            x = layer(x)

        return x

    def __call__(self, X):
        return self.forward(X)

In [ ]:
model = CNN()

In [ ]:
from modules.earlystop import *
from modules.crossentropy import *

train_n = X_train.shape[0]
val_n = X_val.shape[0]

epochs = 50
batch_size = 64
verbose = 1
alpha = 0.1

early_stop = EarlyStop(patience=5)

for epoch in range(epochs):
    indices = np.random.permutation(train_n)
    Xt = X_train[indices]
    yt = y_train[indices]

    for start in range(0, train_n, batch_size):
        end = min(start + batch_size, train_n)
        X_batch = Xt[start:end]
        y_batch = yt[start:end]

        logits_linear = model(X_batch)
        loss = CrossEntropy(logits_linear, y_batch)
        loss.backward()
        loss.grad_descent(alpha)

    val_correct = val_loss = 0
    for start in range(0, val_n, batch_size):
        end = min(start + batch_size, val_n)
        X_batch = X_val[start:end]
        y_batch = y_val[start:end]

        logits_linear = model(X_batch)
        loss = CrossEntropy(logits_linear, y_batch)

        val_loss += loss.value * X_batch.shape[0]
        val_correct += np.sum(logits_linear.x.argmax(1) == y_batch)
    val_loss /= val_n

    stop = early_stop(val_loss)

    if verbose and ((epoch+1)%verbose == 0 or epoch==0) or stop:
        print(f"epoch {epoch+1:02d} | val acc: {val_correct/val_n:.5f}")

    if stop: break

print("done.")

epoch 01 | val acc: 0.96048
epoch 02 | val acc: 0.97452
epoch 03 | val acc: 0.97452
epoch 04 | val acc: 0.98393
epoch 05 | val acc: 0.98286
epoch 06 | val acc: 0.98381
epoch 07 | val acc: 0.98476
epoch 08 | val acc: 0.98488
epoch 09 | val acc: 0.98583
epoch 10 | val acc: 0.98714
epoch 11 | val acc: 0.98476
epoch 12 | val acc: 0.98774
epoch 13 | val acc: 0.98571
epoch 14 | val acc: 0.98702
epoch 15 | val acc: 0.98786
epoch 16 | val acc: 0.98774
epoch 17 | val acc: 0.98726
done.


In [ ]:
X_test = pd.read_csv("data/test.csv").to_numpy(dtype=np.float32)
X_test = X_test.reshape(-1, 1, 28, 28) / 255.0
y_test_cap = model(X_test).x.argmax(axis=1)

res = pd.DataFrame({
    "ImageId": range(1, len(y_test_cap)+1),
    "Label": y_test_cap
})

res.to_csv("submissions/submission_my.csv", index=False)